In [0]:
%pip install langchain langchain-community unstructured langchain-huggingface sentence-transformers faiss-cpu

In [0]:
from langchain_community.document_loaders import (DirectoryLoader,TextLoader)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv("OPENAI_API_KEY") is not None)


loader = DirectoryLoader(
    "knowledge/",
    glob = "**/*.md",
    loader_cls= TextLoader
)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(chunks,embeddings)


retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

question = "What is kills?"

# Retrieval
results = retriever.invoke(question)

# Convert retrieved Documents into context
context = "\n\n".join(
    doc.page_content
    for doc in results
)

# Generation prompt
prompt = f"""
You are a PlayerForge gaming analytics assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{question}

If the answer cannot be found in the context,
say "I don't have enough information to answer that."
"""

# LLM generates the answer
response = llm.invoke(prompt)

print(response.content)

